<a href="https://colab.research.google.com/github/osmium813/Osmiums_Magic/blob/master/dora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import files
uploaded = files.upload()

Saving train.jsonl to train.jsonl


In [2]:
import json
from pathlib import Path

data_path = Path("train.jsonl")
assert data_path.exists(), "train.jsonl was not found."

rows = []
with data_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        for key in ["instruction", "input", "output"]:
            assert key in obj, f"Line {i}: missing {key}"
        assert obj["instruction"].strip(), f"Line {i}: empty instruction"
        assert obj["output"].strip(), f"Line {i}: empty output"
        rows.append(obj)

print(f"OK: {len(rows)} rows")
print(rows[0])

OK: 4 rows
{'instruction': '問い合わせ文を丁寧なビジネスメールに直してください。', 'input': '納期いつですか？早く返事ください。', 'output': 'お世話になっております。納期について確認させていただけますでしょうか。お忙しいところ恐れ入りますが、ご返信いただけますと幸いです。'}


In [9]:
from datasets import Dataset

SYSTEM_MESSAGE = "あなたはユーザーの指示に正確に従う日本語アシスタントです。"

def to_messages(row):
    user_text = row["instruction"].strip()
    if row.get("input", "").strip():
        user_text += "\n\nInput:\n" + row["input"].strip()
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": row["output"].strip()},
        ]
    }

dataset = Dataset.from_list([to_messages(r) for r in rows])
split = dataset.train_test_split(test_size=0.1, seed=42) if len(dataset) >= 20 else {"train": dataset, "test": dataset}
split["train"][0]

{'messages': [{'role': 'system', 'content': 'あなたはユーザーの指示に正確に従う日本語アシスタントです。'},
  {'role': 'user',
   'content': '問い合わせ文を丁寧なビジネスメールに直してください。\n\nInput:\n納期いつですか？早く返事ください。'},
  {'role': 'assistant',
   'content': 'お世話になっております。納期について確認させていただけますでしょうか。お忙しいところ恐れ入りますが、ご返信いただけますと幸いです。'}]}

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [12]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

print(formatting_func(split["train"][0])[:1000])

<bos><start_of_turn>user
あなたはユーザーの指示に正確に従う日本語アシスタントです。

問い合わせ文を丁寧なビジネスメールに直してください。

Input:
納期いつですか？早く返事ください。<end_of_turn>
<start_of_turn>model
お世話になっております。納期について確認させていただけますでしょうか。お忙しいところ恐れ入りますが、ご返信いただけますと幸いです。<end_of_turn>



In [13]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir="gemma3-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=1024,
    logging_steps=5,
    save_steps=50,
    eval_strategy="steps" if len(split["test"]) >= 2 else "no",
    eval_steps=50,
    fp16=True,
    bf16=False,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"] if len(split["test"]) >= 2 else None,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Step,Training Loss,Validation Loss
2,No log,4.208662


TrainOutput(global_step=2, training_loss=5.4145402908325195, metrics={'train_runtime': 10.0408, 'train_samples_per_second': 0.797, 'train_steps_per_second': 0.199, 'total_flos': 2533796660736.0, 'train_loss': 5.4145402908325195})

In [14]:
trainer.save_model("gemma3-lora-adapter")
tokenizer.save_pretrained("gemma3-lora-adapter")

('gemma3-lora-adapter/tokenizer_config.json',
 'gemma3-lora-adapter/chat_template.jinja',
 'gemma3-lora-adapter/tokenizer.json')

In [16]:
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": "問い合わせ文を丁寧なビジネスメールに直してください。\n\n入力:\n納期まだ？急いで。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

outputs = trainer.model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

件名：納期について

〇〇様

いつもお世話になっております。

この度は、〇〇の件につきまして、納期についてご心配とのこと、大変申し訳ございません。

現在、状況を詳細に確認させていただきており、改めてご提示いただくことと、可能な範囲で対応を検討させていただきます。

お忙しいところ恐れ入りますが、何卒よろしくお願い申し上げます。

敬具

[あなたの名前]
[あなたの役職]

---

**ポイント:**

*   **件名:** 問い合わせ内容を明確にする
*   **
